# Bodhan Indic-Translate QLoRA: Dehwali Bhili → Marathi

Runs the full pipeline on a Colab **T4 GPU**: setup, data preparation, a training smoke test, resumable training, and test-set evaluation. All artifacts are written to Google Drive under `MyDrive/BodhanAI/artifacts/bodhan-bhili-mt/<run_id>/`.

## Part A: Setup (every new Colab session)

Runtime → Change runtime type → **T4 GPU**. Add a Hugging Face read token named `HF_TOKEN` to Colab Secrets (key icon) and accept the model's terms on Hugging Face.

In [ ]:
# Cell 1: Mount Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Cell 2: Clone the latest code
# Only in a fresh session: this deletes the repo folder, including the downloaded model cache.
%cd /content
!rm -rf BodhanAI-Bhili-to-Marathi-MT
!git clone https://github.com/Dhwani9Dhingra/BodhanAI-Bhili-to-Marathi-MT.git
%cd /content/BodhanAI-Bhili-to-Marathi-MT
!git log --oneline -1

In [ ]:
# Cell 3: Install dependencies
!pip install -U pip
!pip install -r requirements/colab.txt

In [ ]:
# Cell 4: Token and environment
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ.pop("BODHAN_RUN_ID", None)  # would override the run_id in the config
print("HF token available:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# Cell 5: Preflight and data preparation for the main run (bhili_marathi_t4_v1)
# Replace the path with the location of the raw TSV on your Drive. Run once per run_id.
DATA_TSV = "/content/drive/MyDrive/BodhanAI/data/Dehwali_Bhili_Translation_15k_pipeline.tsv"

!python -m scripts.preflight --config configs/colab_t4.yaml
!python -m scripts.prepare_data --config configs/colab_t4.yaml --data "{DATA_TSV}"
# Note the printed "Frozen test SHA-256"; every later stage checks it.

## Part B: Training smoke test (`configs/smoke.yaml`)

10 optimizer steps, a checkpoint every 5 steps, dev loss on 20 examples every 5 steps. Uses its own run folder (`smoke_001`), so it never touches the main run. It also tests resuming: stop the training cell after the first checkpoint, then resume.

In [ ]:
# Cell 6: Preflight and data preparation for smoke_001
!python -m scripts.preflight --config configs/smoke.yaml
!python -m scripts.prepare_data --config configs/smoke.yaml --data "{DATA_TSV}"

In [ ]:
# Cell 7: Start training, then interrupt it on purpose
# When the log shows "Checkpoint saved: step 5/10", press Stop to simulate a disconnect.
# The log should end with "Training failed: ..." and "Re-run the same command to resume...".
!python -m scripts.train --config configs/smoke.yaml

In [ ]:
# Cell 8: Check the checkpoint on Drive
# Expect adapter_model.safetensors, adapter_config.json, optimizer.pt, scheduler.pt, trainer_state.json.
!ls -la /content/drive/MyDrive/BodhanAI/artifacts/bodhan-bhili-mt/smoke_001/checkpoints/trainer/checkpoint-5
!cat /content/drive/MyDrive/BodhanAI/artifacts/bodhan-bhili-mt/smoke_001/reports/training_progress.json

In [ ]:
# Cell 9: Resume
# Passes if the log shows "Resuming from checkpoint-5 (step 5/10)", training continues from step 5,
# and it ends with "TRAINING STATUS: COMPLETED" and "Optimizer steps: 10".
!python -m scripts.train --config configs/smoke.yaml

In [ ]:
# Cell 10: Check the outputs
run = "/content/drive/MyDrive/BodhanAI/artifacts/bodhan-bhili-mt/smoke_001"
!ls {run}/checkpoints/trainer {run}/checkpoints/adapter_best {run}/checkpoints/adapter_final
!head -c 1500 {run}/reports/training_report.json
# Each smoke run takes about 10-15 minutes, mostly downloading and loading the model.

## Part C: Full training (`configs/colab_t4.yaml`)

`max_steps: 1400` in the config is one pass over the ~11,195 training examples (effective batch 8), about 6.5 hours on a T4 including dev evaluations. Checkpoints go to Drive every 25 steps.

- **If the runtime disconnects:** run cells 1-4 (skip Cell 5), then Cell 11 again unchanged. It resumes from the newest complete checkpoint.
- Do not change `max_steps` or other training settings mid-run; resuming refuses a changed configuration.
- Do not rerun `prepare_data` for this run_id once training has started.

In [ ]:
# Cell 11: Full training (resumable)
!python -m scripts.train --config configs/colab_t4.yaml

# Progress (also readable from Drive while training runs):
!cat /content/drive/MyDrive/BodhanAI/artifacts/bodhan-bhili-mt/bhili_marathi_t4_v1/reports/training_progress.json

In [ ]:
# Cell 12 (only when extending a finished run): continue training after raising max_steps
# This project first trained 700 steps, then raised max_steps to 1400 and continued with --extend.
# The pre-extension adapter is preserved in checkpoints/adapter_step_<N>/.
!python -m scripts.train --config configs/colab_t4.yaml --extend

In [ ]:
# Cell 13: When training has finished, flush pending Drive uploads.
# Wait for this to finish before closing the tab: unsynced checkpoints are lost with the runtime.
# Re-run Cell 1 afterwards if you continue in this session.
drive.flush_and_unmount()

## Part D: Evaluation setup

Needs Drive mounted (Cell 1). If this session was cloned before the evaluation code existed, pull instead of re-cloning so the downloaded model cache is kept.

In [ ]:
# Cell 14: Update the code without deleting the model cache
%cd /content/BodhanAI-Bhili-to-Marathi-MT
!git pull
!ls scripts/evaluate.py

In [ ]:
# Cell 15: Confirm the final checkpoint and existing predictions
run = "/content/drive/MyDrive/BodhanAI/artifacts/bodhan-bhili-mt/bhili_marathi_t4_v1"
!cat {run}/checkpoints/adapter_best/best_metric.json
!ls {run}/evaluation

## Part E: Test-set evaluation

Every system translates the same 300 sentences sampled (seed 42) from the frozen test split. Predictions are saved batch by batch, so re-running a command after a disconnect continues where it stopped.

| System | Model |
| --- | --- |
| `base` | Untuned Bodhan |
| `tuned_700` | Adapter after 700 steps (`checkpoints/adapter_step_701`) |
| `tuned` | Final adapter (`checkpoints/adapter_best`), packaged |

In [ ]:
# Cell 16: Evaluation smoke test (5 sentences). Check the "sent/s" and "ETA" lines.
!python -m scripts.evaluate generate --config configs/colab_t4.yaml --system base --limit 5

In [ ]:
# Cell 17: Base model on all 300 sentences (reuses the 5 above)
!python -m scripts.evaluate generate --config configs/colab_t4.yaml --system base

In [ ]:
# Cell 18: Sanity-check the base predictions (no GPU)

import json, collections

run = "/content/drive/MyDrive/BodhanAI/artifacts/bodhan-bhili-mt/bhili_marathi_t4_v1"
rows = [json.loads(l) for l in open(f"{run}/evaluation/predictions_base.jsonl", encoding="utf-8") if l.strip()]

def deva(t):  # share of letters that are Devanagari
    letters = [c for c in t if c.isalpha()]
    return sum("\u0900" <= c <= "\u097F" for c in letters) / max(len(letters), 1)

ids = [r["record_id"] for r in rows]
print("rows:", len(rows), "| unique ids:", len(set(ids)), "| duplicates:", len(ids) - len(set(ids)))
print("empty:", sum(not r["prediction"].strip() for r in rows))
print("exact copy of source:", sum(r["prediction"].strip() == r["source"].strip() for r in rows))
print("exact match with reference:", sum(r["prediction"].strip() == r["target"].strip() for r in rows))
print("hit token limit:", sum(r["hit_token_limit"] for r in rows))
print("mean Devanagari share:", round(sum(deva(r["prediction"]) for r in rows) / len(rows), 3))
print("pred/ref length ratio:", round(sum(len(r["prediction"]) for r in rows) / sum(len(r["target"]) for r in rows), 2))
print(json.load(open(f"{run}/evaluation/generation_base.json")))
for r in rows[::60]:
    print("\nSRC :", r["source"]); print("REF :", r["target"]); print("PRED:", r["prediction"])

In [ ]:
# Cell 19: 700-step adapter
!python -m scripts.evaluate generate --config configs/colab_t4.yaml --system tuned_700 --adapter checkpoints/adapter_step_701

In [ ]:
# Cell 20: Final adapter
!python -m scripts.evaluate generate --config configs/colab_t4.yaml --system tuned

In [ ]:
# Cell 21: Scores, significance tests, analysis and package (no GPU needed)
!python -m scripts.evaluate report --config configs/colab_t4.yaml

In [ ]:
# Cell 22: Flush all results to Drive; wait for it to finish before closing the tab
drive.flush_and_unmount()